In [1]:
from pyspark.sql import SparkSession

# Configuración mínima (modo local)
spark = (SparkSession.builder
    .appName("lectura_escritura")
    .master("local[*]")       # [*] = todos los cores disponibles
    .getOrCreate())

# Verificar versión
print(spark.version)            # 3.x.x
print(spark.sparkContext.uiWebUrl)  # http://localhost:4040

4.1.1
http://55070f6a127f:4040


# Ejercicio 1

**Realiza el siguiente ejercicio de auditoria de calidad de datos**

In [6]:
# Dataset con problemas
datos_audit = spark.createDataFrame([
    ("U001", "Ana",    "F", 28, 72000.0, "CDMX",  "2023-01-10"),
    ("U002", "Carlos", "M", None, 55000.0, "GDL",  "2023-02-15"),
    ("U003", "Maria",  "F", 35, None,    "MTY",  "2023-03-20"),
    ("U004", "Juan",   "M", 45, 88000.0, None,   "2023-01-10"),
    ("U005", "Laura",  "F", 28, 72000.0, "CDMX",  "2023-01-10"),  # Duplicado de U001
    ("U006", "Pedro",  "M", -5, 95000.0, "CDMX",  "N/A"),
    ("U007", None,     "M", 30, 61000.0, "GDL",  "2023-04-05"),
    ("U001", "Ana",    "F", 28, 72000.0, "CDMX",  "2023-01-10"),  # Duplicado exacto
], ["user_id", "nombre", "genero", "edad", "salario", "ciudad", "fecha_registro"])

datos_audit.show()

+-------+------+------+----+-------+------+--------------+
|user_id|nombre|genero|edad|salario|ciudad|fecha_registro|
+-------+------+------+----+-------+------+--------------+
|   U001|   Ana|     F|  28|72000.0|  CDMX|    2023-01-10|
|   U002|Carlos|     M|NULL|55000.0|   GDL|    2023-02-15|
|   U003| Maria|     F|  35|   NULL|   MTY|    2023-03-20|
|   U004|  Juan|     M|  45|88000.0|  NULL|    2023-01-10|
|   U005| Laura|     F|  28|72000.0|  CDMX|    2023-01-10|
|   U006| Pedro|     M|  -5|95000.0|  CDMX|           N/A|
|   U007|  NULL|     M|  30|61000.0|   GDL|    2023-04-05|
|   U001|   Ana|     F|  28|72000.0|  CDMX|    2023-01-10|
+-------+------+------+----+-------+------+--------------+



Realizar:

- Genera el reporte de % de nulls por columna
- Identifica y cuenta duplicados exactos y parciales (por user_id)
- Detecta valores inválidos: edad negativa, fecha = "N/A"
- Crea un DataFrame limpio resolviendo todos los problemas encontrados
- Genera un reporte comparativo: filas_originales, filas_eliminadas, filas_finales, % retención

In [7]:
from pyspark.sql import functions as F
# 1. Reporte de Null por columna
total_filas = datos_audit.count()

null_report = datos_audit.select([
    (
        F.sum(F.col(c).isNull().cast("int")) / total_filas * 100
    ).alias(c)
    for c in datos_audit.columns
])

print("Porcentaje de nulls por columna:")
null_report.show()

Porcentaje de nulls por columna:
+-------+------+------+----+-------+------+--------------+
|user_id|nombre|genero|edad|salario|ciudad|fecha_registro|
+-------+------+------+----+-------+------+--------------+
|    0.0|  12.5|   0.0|12.5|   12.5|  12.5|           0.0|
+-------+------+------+----+-------+------+--------------+



In [8]:
# 2. Duplicados Exactos

duplicados_exactos = total_filas - datos_audit.dropDuplicates().count()

print(f"Duplicados exactos: {duplicados_exactos}")

Duplicados exactos: 1


In [9]:
# 3. Duplicados parciales (user_id)

duplicados_user = (
    datos_audit.groupBy("user_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicados parciales por user_id:")
duplicados_user.show()

cantidad_dup_user = duplicados_user.count()

print(f"Cantidad de user_id duplicados: {cantidad_dup_user}")


Duplicados parciales por user_id:
+-------+-----+
|user_id|count|
+-------+-----+
|   U001|    2|
+-------+-----+

Cantidad de user_id duplicados: 1


In [10]:
# 4. Valores Invalidos

edad_invalida = datos_audit.filter(F.col("edad") < 0)

fecha_invalida = datos_audit.filter(F.col("fecha_registro") == "N/A")

print("Edades invalidas:")
edad_invalida.show()

print("Fechas invalidas:")
fecha_invalida.show()

Edades invalidas:
+-------+------+------+----+-------+------+--------------+
|user_id|nombre|genero|edad|salario|ciudad|fecha_registro|
+-------+------+------+----+-------+------+--------------+
|   U006| Pedro|     M|  -5|95000.0|  CDMX|           N/A|
+-------+------+------+----+-------+------+--------------+

Fechas invalidas:
+-------+------+------+----+-------+------+--------------+
|user_id|nombre|genero|edad|salario|ciudad|fecha_registro|
+-------+------+------+----+-------+------+--------------+
|   U006| Pedro|     M|  -5|95000.0|  CDMX|           N/A|
+-------+------+------+----+-------+------+--------------+



In [11]:
# 5. Limpieza del DF

df_limpio = (
    datos_audit
    .dropDuplicates()                           # elimina duplicados exactos
    .dropDuplicates(["user_id"])                # elimina duplicados por user_id
    .filter(F.col("edad") >= 0)                 # elimina edades negativas
    .filter(F.col("fecha_registro") != "N/A")   # elimina fechas invalidas
)

print("DataFrame limpio:")
df_limpio.show()

DataFrame limpio:
+-------+------+------+----+-------+------+--------------+
|user_id|nombre|genero|edad|salario|ciudad|fecha_registro|
+-------+------+------+----+-------+------+--------------+
|   U001|   Ana|     F|  28|72000.0|  CDMX|    2023-01-10|
|   U003| Maria|     F|  35|   NULL|   MTY|    2023-03-20|
|   U004|  Juan|     M|  45|88000.0|  NULL|    2023-01-10|
|   U005| Laura|     F|  28|72000.0|  CDMX|    2023-01-10|
|   U007|  NULL|     M|  30|61000.0|   GDL|    2023-04-05|
+-------+------+------+----+-------+------+--------------+



In [12]:
# 6. Reporte

filas_originales = total_filas
filas_finales = df_limpio.count()
filas_eliminadas = filas_originales - filas_finales

retencion = (filas_finales / filas_originales) * 100

reporte = spark.createDataFrame([
    (
        filas_originales,
        filas_eliminadas,
        filas_finales,
        round(retencion, 2)
    )
], [
    "filas_originales",
    "filas_eliminadas",
    "filas_finales",
    "%_retencion"
])

print("Reporte comparativo:")
reporte.show()

Reporte comparativo:
+----------------+----------------+-------------+-----------+
|filas_originales|filas_eliminadas|filas_finales|%_retencion|
+----------------+----------------+-------------+-----------+
|               8|               3|            5|       62.5|
+----------------+----------------+-------------+-----------+



# Ejercicio 2

**Utilizando los archivos de la carpeta 'Datos_all' realiza lo siguiente:**

Realizar:

- Lee vuelos_sucio.csv SIN usar inferSchema, con esquema explícito y nullValue="N/A"
- Lee eventos.json y explora el campo tags para tener una fila por etiqueta por vuelo
- Escribe el resultado limpio como Parquet particionado por aerolinea
- Verifica que al leer el Parquet y filtrar por aerolinea = "AM", solo se lean los archivos de AM (revisar en Spark UI)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# 1. Leer vuelos_sucio.csv SIN usar inferSchema, con esquema expl�cito y nullValue="N/A"
schema_vuelos = StructType([
    StructField("vuelo_id",    StringType(),    nullable=False),
    StructField("fecha",       DateType(),      nullable=True),
    StructField("origen",      StringType(),    nullable=True),
    StructField("destino",     StringType(),    nullable=True),
    StructField("aerolinea",   StringType(),    nullable=True),
    StructField("pasajeros",   IntegerType(),   nullable=True),
    StructField("delay_min",   IntegerType(),   nullable=True),
    StructField("ingreso_usd", DoubleType(),    nullable=True),
])

df_vuelos = (spark.read
    .option("header", "true")
    .option("nullValue", "N/A")
    .schema(schema_vuelos)
    .csv("/home/jovyan/work/Datos_all/csv_sucio/vuelos_sucio.csv")
)

df_vuelos.show(5)



+--------+----------+------+-------+---------+---------+---------+-----------+
|vuelo_id|     fecha|origen|destino|aerolinea|pasajeros|delay_min|ingreso_usd|
+--------+----------+------+-------+---------+---------+---------+-----------+
| MX99001|      NULL|  CDMX|    CUN|       AM|      150|       45|     1850.0|
| MX99002|2023-03-10|   GDL|    LAX|       Y4|     NULL|       30|     2100.5|
| MX99003|2023-04-22|   MTY|    MEX|       UA|       90|        0|       NULL|
| MX99004|2023-05-01|   CUN|    ORD|     AA  |      175|       90|     2750.0|
| MX99005|2023-06-15|  CDMX|    JFK|       AM|       -5|      120|     4100.0|
+--------+----------+------+-------+---------+---------+---------+-----------+
only showing top 5 rows


In [3]:
# 2. Lee eventos.json y explora el campo tags para tener una fila por etiqueta por vuelo
df_eventos = spark.read.json("/home/jovyan/work/Datos_all/json/eventos.json")

df_eventos_exploded = df_eventos.select(
    "vuelo_id",
    "evento_id",
    "tipo",
    "hora",
    "severidad",
    F.explode("tags").alias("tag"),
    "metadata.*"
)

df_eventos_exploded.show(5)



+--------+---------+-----------+-----+---------+---------+----+--------+
|vuelo_id|evento_id|       tipo| hora|severidad|      tag|gate|terminal|
+--------+---------+-----------+-----+---------+---------+----+--------+
| MX02048|  E388389|   despegue|12:14|     info|  puntual| E14|       1|
| MX02048|  E388389|   despegue|12:14|     info|codeshare| E14|       1|
| MX02048|  E388389|   despegue|12:14|     info| low_cost| E14|       1|
| MX02048|  E131244|gate_change|07:13|     info|codeshare| E14|       2|
| MX02048|  E131244|gate_change|07:13|     info|  puntual| E14|       2|
+--------+---------+-----------+-----+---------+---------+----+--------+
only showing top 5 rows


In [4]:
# Unir para tener el resultado limpio (vuelos + eventos explotados)
# El ejercicio pide "Escribe el resultado limpio", implica la combinacion o el procesamiento final.
# Dado que pide particionar por aerolinea, necesitamos la aerolinea en el DF final.
df_final = df_eventos_exploded.join(df_vuelos, "vuelo_id", "inner")

# 3. Escribe el resultado limpio como Parquet particionado por aerolinea
output_path = "vuelos_eventos_limpio"
df_final.write \
    .mode("overwrite") \
    .partitionBy("aerolinea") \
    .parquet(output_path)

# 4. Verifica que al leer el Parquet y filtrar por aerolinea = "AM", solo se lean los archivos de AM
df_verificacion = spark.read.parquet(output_path) \
    .filter(F.col("aerolinea") == "AM")

df_verificacion.show()

+--------+---------+-----------+-----+---------+--------+----+--------+----------+------+-------+---------+---------+-----------+---------+
|vuelo_id|evento_id|       tipo| hora|severidad|     tag|gate|terminal|     fecha|origen|destino|pasajeros|delay_min|ingreso_usd|aerolinea|
+--------+---------+-----------+-----+---------+--------+----+--------+----------+------+-------+---------+---------+-----------+---------+
| MX02058|  E392004|   despegue|20:13|  critico|low_cost| E19|       4|2023-01-15|   TIJ|    JFK|      134|      -13|    1788.76|       AM|
| MX02058|  E354801| aterrizaje|20:51|     warn| puntual| A22|       4|2023-01-15|   TIJ|    JFK|      134|      -13|    1788.76|       AM|
| MX02059|  E589710|   boarding|06:43|  critico| puntual| C26|       1|2023-07-28|  CDMX|    CUN|      188|       76|    1548.75|       AM|
| MX02059|  E589710|   boarding|06:43|  critico|low_cost| C26|       1|2023-07-28|  CDMX|    CUN|      188|       76|    1548.75|       AM|
| MX02059|  E589710|

* 5. Verifica que al leer el Parquet y filtrar por aerolinea = "AM", solo se lean los archivos de AM (revisar en Spark UI)

== Physical Plan ==
CollectLimit (4)
+- * Project (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [15]: [vuelo_id#108, evento_id#109, tipo#110, hora#111, severidad#112, tag#113, gate#114, terminal#115, fecha#116, origen#117, destino#118, pasajeros#119, delay_min#120, ingreso_usd#121, aerolinea#122]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/vuelos_eventos_limpio]
PartitionFilters: [isnotnull(aerolinea#122), (aerolinea#122 = AM)]
ReadSchema: struct<vuelo_id:string,evento_id:string,tipo:string,hora:string,severidad:string,tag:string,gate:string,terminal:string,fecha:date,origen:string,destino:string,pasajeros:int,delay_min:int,ingreso_usd:double>

(2) ColumnarToRow [codegen id : 1]
Input [15]: [vuelo_id#108, evento_id#109, tipo#110, hora#111, severidad#112, tag#113, gate#114, terminal#115, fecha#116, origen#117, destino#118, pasajeros#119, delay_min#120, ingreso_usd#121, aerolinea#122]

(3) Project [codegen id : 1]
Output [15]: [toprettystring(vuelo_id#108, Some(Etc/UTC)) AS vuelo_id#124, toprettystring(evento_id#109, Some(Etc/UTC)) AS evento_id#125, toprettystring(tipo#110, Some(Etc/UTC)) AS tipo#126, toprettystring(hora#111, Some(Etc/UTC)) AS hora#127, toprettystring(severidad#112, Some(Etc/UTC)) AS severidad#128, toprettystring(tag#113, Some(Etc/UTC)) AS tag#129, toprettystring(gate#114, Some(Etc/UTC)) AS gate#130, toprettystring(terminal#115, Some(Etc/UTC)) AS terminal#131, toprettystring(fecha#116, Some(Etc/UTC)) AS fecha#132, toprettystring(origen#117, Some(Etc/UTC)) AS origen#133, toprettystring(destino#118, Some(Etc/UTC)) AS destino#134, toprettystring(pasajeros#119, Some(Etc/UTC)) AS pasajeros#135, toprettystring(delay_min#120, Some(Etc/UTC)) AS delay_min#136, toprettystring(ingreso_usd#121, Some(Etc/UTC)) AS ingreso_usd#137, toprettystring(aerolinea#122, Some(Etc/UTC)) AS aerolinea#138]
Input [15]: [vuelo_id#108, evento_id#109, tipo#110, hora#111, severidad#112, tag#113, gate#114, terminal#115, fecha#116, origen#117, destino#118, pasajeros#119, delay_min#120, ingreso_usd#121, aerolinea#122]

(4) CollectLimit
Input [15]: [vuelo_id#124, evento_id#125, tipo#126, hora#127, severidad#128, tag#129, gate#130, terminal#131, fecha#132, origen#133, destino#134, pasajeros#135, delay_min#136, ingreso_usd#137, aerolinea#138]
Arguments: 21